In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Try to enable TensorFlow path; if not available we fall back later.
USE_TF = True
try:
    from tensorflow.keras import Input
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense
except Exception:
    USE_TF = False

from sklearn.ensemble import RandomForestRegressor

# Personal health risk bonuses
RISK_BONUS = {
    "Asthma": 30,
    "COPD": 35,
    "Chronic bronchitis": 25,
    "Cardiovascular disease": 20,
    "Smoker": 15,
    "Elderly (65+)": 10,
    "Child (<12)": 10,
}

def base_prescription(aqi: float) -> str:
    if aqi < 75:
        return "Low risk: Safe for all activities."
    elif aqi < 125:
        return "Moderate risk: Sensitive groups should limit outdoor exertion."
    elif aqi < 175:
        return "High risk: Reduce outdoor exertion. Masks recommended."
    else:
        return "Very high risk: Avoid all outdoor exertion. Stay indoors."

def personalize(aqi: float, selected_conditions) -> tuple[float, int, str]:
    bonus = sum(RISK_BONUS.get(c, 0) for c in selected_conditions)
    personalized = aqi + bonus
    phrs = int(min(100, round((personalized/300)*100)))
    return personalized, phrs, base_prescription(personalized)

def standardize_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.str.strip()
    if 'State' in df.columns and 'District' in df.columns:
        df['State'] = df['State'].astype(str).str.strip().str.lower()
        df['District'] = df['District'].astype(str).str.strip().str.lower()
    return df

DISTRICT_FIX = {
    'ahmadabad': 'ahmedabad','aizwal':'aizawl','anatapur':'anantapur',
    'ananthapuramu':'anantapur','bangalore':'bengaluru','belgaum':'belagavi',
    'calcutta':'kolkata','cochin':'kochi','ysr kadapa':'kadapa',
    'east khasi hills':'shillong','gurgaon':'gurugram','hubli':'dharwad',
    'jullundur':'jalandhar','madras':'chennai','mumbai':'greater mumbai',
    'mysore':'mysuru','new delhi':'delhi','north goa':'goa',
    'south goa':'goa','ponda':'goa','poona':'pune',
    'trivandrum':'thiruvananthapuram','visakhapatnam':'vishakhapatnam',
    'waltair':'vishakhapatnam','warangal (urban)':'warangal'
}


In [22]:
# Load
aqi_df = pd.read_csv('aqidata_with_state_district.csv')
temp_df = pd.read_csv('tempdata.csv')
rainfall_df = pd.read_csv('rainfalldata.csv')
wind_df = pd.read_csv('winddata.csv')

# Clean + standardize + fix district aliases
aqi_df = standardize_df(aqi_df)
temp_df = standardize_df(temp_df)
rainfall_df = standardize_df(rainfall_df)
wind_df = standardize_df(wind_df)

for _df in [aqi_df, temp_df, rainfall_df, wind_df]:
    if 'District' in _df.columns:
        _df['District'] = _df['District'].replace(DISTRICT_FIX)

# Aggregations per (State, District)
aqi_agg = aqi_df.groupby(['State','District'])['Index Value'].mean().reset_index().rename(columns={'Index Value':'Avg AQI'})

temp_agg = temp_df.groupby(['State','District'])['Temperature (in °C)'].agg(['mean','min','max']).reset_index()
temp_agg.columns = ['State','District','Avg Temp','Min Temp','Max Temp']

rain_agg = rainfall_df.groupby(['State','District'])['Rainfall (mm)'].agg(['mean','max']).reset_index()
rain_agg.columns = ['State','District','Avg Rainfall','Max Rainfall']

wind_agg = wind_df.groupby(['State','District'])['Speed (in m/s)'].agg(['mean','min','max']).reset_index()
wind_agg.columns = ['State','District','Avg Wind Speed','Min Wind','Max Wind']

merged = aqi_agg.merge(temp_agg, on=['State','District'], how='outer') \
                .merge(rain_agg, on=['State','District'], how='outer') \
                .merge(wind_agg, on=['State','District'], how='outer')

# Fill missing with column means
for c in ['Avg AQI','Avg Temp','Min Temp','Max Temp','Avg Rainfall','Max Rainfall','Avg Wind Speed','Min Wind','Max Wind']:
    merged[c] = merged[c].fillna(merged[c].mean())

merged.head()


,State,District,Avg AQI,Avg Temp,Min Temp,Max Temp,Avg Rainfall,Max Rainfall,Avg Wind Speed,Min Wind,Max Wind
0,andaman and nicobar islands,nicobar,120.958285,24.363333,23.510000,24.870000,207.625000,458.20000,2.249381,1.561752,3.117942
1,andaman and nicobar islands,north & middle andaman,120.958285,22.090000,21.740000,22.470000,123.809804,350.12375,2.249381,1.561752,3.117942
2,andaman and nicobar islands,south andaman,120.958285,24.841176,23.660000,25.640000,294.530000,627.80000,2.249381,1.561752,3.117942
3,andhra pradesh,alluri sitharama raju,120.958285,19.871812,14.110939,24.331315,123.809804,350.12375,1.366667,0.900000,1.800000
4,andhra pradesh,anantapur,66.924706,22.140000,19.060000,24.230000,70.470000,134.00000,2.925000,1.800000,5.400000


In [23]:
from sklearn.preprocessing import MinMaxScaler

# Features for the model
features = ['Avg AQI','Avg Temp','Avg Rainfall','Avg Wind Speed']

# === Path A: CNN–LSTM sequence model if TensorFlow is available and we have enough rows ===
def build_and_run_cnn_lstm(df: pd.DataFrame, time_step: int = 5):
    data = df[features].values
    scaler = MinMaxScaler((0,1))
    scaled = scaler.fit_transform(data)

    def make_seq(dataset, t):
        X, y = [], []
        for i in range(len(dataset) - t):
            X.append(dataset[i:i+t, :])
            y.append(dataset[i+t, 0])  # next-step AQI
        return np.array(X), np.array(y)

    X, y = make_seq(scaled, time_step)
    if len(X) < 10:
        raise RuntimeError("Not enough rows for sequence modeling.")

    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, shuffle=False)

    # Use Input(...) to avoid Keras warning
    num_features = X.shape[2]
    model = Sequential([
        Input(shape=(time_step, num_features)),
        Conv1D(64, 2, activation='relu'),
        MaxPooling1D(2),
        LSTM(50, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    model.fit(Xtr, ytr, epochs=50, batch_size=16, validation_data=(Xte, yte), verbose=0)

    pred_scaled = model.predict(Xte, verbose=0)
    dummy = np.zeros((len(pred_scaled), scaled.shape[1]))
    dummy[:, 0] = pred_scaled[:, 0]
    pred_aqi = scaler.inverse_transform(dummy)[:, 0]

    rows = len(pred_aqi)
    res = merged.iloc[-rows:].copy()
    res['Predicted AQI'] = pred_aqi
    return res, "CNN-LSTM (TensorFlow)"

# === Path B: RandomForest fallback (tabular, no TF) ===
def run_random_forest(df: pd.DataFrame):
    X_tab = df[['Avg Temp','Avg Rainfall','Avg Wind Speed']].values
    y_tab = df['Avg AQI'].values
    Xtr, Xte, ytr, yte = train_test_split(X_tab, y_tab, test_size=0.2, random_state=42)
    rf = RandomForestRegressor(n_estimators=300, random_state=42)
    rf.fit(Xtr, ytr)
    pred = rf.predict(Xte)
    rows = len(pred)
    res = merged.iloc[-rows:].copy()
    res['Predicted AQI'] = pred
    return res, "RandomForest fallback"

# Decide which path to use
try:
    if USE_TF:
        out, model_used = build_and_run_cnn_lstm(merged, time_step=5)
    else:
        out, model_used = run_random_forest(merged)
except Exception:
    out, model_used = run_random_forest(merged)

# Prescriptions and CHIS
out['Prescription'] = out['Predicted AQI'].apply(base_prescription)

for c in ['Avg AQI','Avg Temp','Avg Rainfall','Avg Wind Speed']:
    mn, mx = out[c].min(), out[c].max()
    out[f'norm_{c}'] = (out[c] - mn) / (mx - mn) if mx > mn else 0.0

weights = {'aqi':0.5,'temp':0.2,'rainfall':0.1,'wind':0.2}
out['CHIS'] = (weights['aqi']*out['norm_Avg AQI'] +
               weights['temp']*out['norm_Avg Temp'] +
               weights['rainfall']*out['norm_Avg Rainfall'] +
               weights['wind']*out['norm_Avg Wind Speed'])

# Keys for lookup
out['State_key'] = out['State'].astype(str).str.strip().str.lower()
out['District_key'] = out['District'].astype(str).str.strip().str.lower()

final_results = out[['State','District','Avg AQI','Avg Temp','Min Temp','Max Temp',
                     'Avg Rainfall','Max Rainfall','Avg Wind Speed','Min Wind','Max Wind',
                     'CHIS','Predicted AQI','Prescription']].copy()

print("Model used:", model_used)
display(final_results.head())


Model used: CNN-LSTM (TensorFlow)


,State,District,Avg AQI,Avg Temp,Min Temp,Max Temp,Avg Rainfall,Max Rainfall,Avg Wind Speed,Min Wind,Max Wind,CHIS,Predicted AQI,Prescription
643,telangana,hyderabad,91.798212,20.115714,17.240000,22.850000,90.891667,268.20000,2.933333,1.9,5.1,0.388740,127.467540,High risk: Reduce outdoor exertion. Masks reco...
644,telangana,jagitial,120.958285,19.871812,14.110939,24.331315,123.809804,350.12375,2.283333,1.5,3.4,0.450090,113.326951,Moderate risk: Sensitive groups should limit o...
645,telangana,jangaon,120.958285,19.871812,14.110939,24.331315,123.809804,350.12375,2.966667,2.0,4.9,0.501180,124.008047,Moderate risk: Sensitive groups should limit o...
646,telangana,jayashankar bhupalapally,120.958285,19.871812,14.110939,24.331315,123.809804,350.12375,1.925000,1.4,2.6,0.423298,120.773421,Moderate risk: Sensitive groups should limit o...
647,telangana,jogulamba gadwal,120.958285,19.871812,14.110939,24.331315,123.809804,350.12375,3.050000,2.1,4.6,0.507411,127.084258,High risk: Reduce outdoor exertion. Masks reco...


In [24]:
# Cell 5 — Interactive input version (no widgets, just input prompts)

from IPython.display import display, HTML

# Simple console prompts for user input
print("=== Air Quality Prediction System ===")

# List available states (first 10 for reference)
states_available = sorted(out['State'].dropna().unique().tolist())
print("\nAvailable example states (first 10):")
print(", ".join(states_available[:10]))

# --- Get inputs ---
state_in = input("\nEnter your State: ").strip().lower()
district_in = input("Enter your District (press Enter to skip): ").strip().lower()

# Show available health conditions
print("\nHealth Conditions you can choose from:")
for i, cond in enumerate(RISK_BONUS.keys(), start=1):
    print(f"{i}. {cond}")

cond_nums = input("\nEnter condition numbers separated by commas (or press Enter for none): ").strip()
conditions = []
if cond_nums:
    for num in cond_nums.split(","):
        try:
            idx = int(num.strip()) - 1
            key = list(RISK_BONUS.keys())[idx]
            conditions.append(key)
        except (ValueError, IndexError):
            pass

print("\nYou selected:")
print("State:", state_in or "(none)")
print("District:", district_in or "(none)")
print("Health Conditions:", ", ".join(conditions) if conditions else "None")

# --- Run prediction logic ---
sel = out[out['State_key'] == state_in].copy()
if district_in:
    sel = sel[sel['District_key'] == district_in].copy()

if sel.empty:
    print(f"\nNo matching rows for state='{state_in}', district='{district_in}'.")
    print("Try one of these states:", states_available[:10])
else:
    row = sel.iloc[0]
    aqi_pred = float(row['Predicted AQI'])
    personalized_aqi, phrs, presc = personalize(aqi_pred, conditions)

    min_temp, max_temp = float(row['Min Temp']), float(row['Max Temp'])
    min_wind, max_wind = float(row['Min Wind']), float(row['Max Wind'])
    rainfall_label = "Rain expected" if float(row['Avg Rainfall']) > 0.1 else "No significant rainfall"

    html = f"""
    <div style="font-family:system-ui,Segoe UI,Roboto,Arial;line-height:1.45">
      <h3 style="margin:0 0 8px 0">Personalized AQI Result</h3>
      <div><b>Location:</b> {row['State'].title()}, {row['District'].title() if row['District'] else '(district not specified)'}</div>
      <div><b>Selected Conditions:</b> {', '.join(conditions) if conditions else 'None'}</div>
      <div><b>Predicted AQI:</b> {aqi_pred:.1f}</div>
      <div><b>Personalized AQI:</b> {personalized_aqi:.1f} &nbsp; <b>PHRS:</b> {phrs}</div>
      <div><b>Prescription:</b> {presc}</div>
      <hr style="margin:10px 0">
      <h4 style="margin:0 0 8px 0">Local Conditions</h4>
      <div><b>Temp range (°C):</b> {min_temp:.1f} – {max_temp:.1f}</div>
      <div><b>Wind range (m/s):</b> {min_wind:.1f} – {max_wind:.1f}</div>
      <div><b>Rainfall:</b> {rainfall_label}</div>
    </div>
    """
    display(HTML(html))

    # Show a quick preview of the district record + the whole dataset
    display(sel[['State','District','Avg AQI','Avg Temp','Avg Rainfall','Avg Wind Speed','Predicted AQI','CHIS','Prescription']].head(1))
    display(HTML("<hr>"))
    display(HTML("<b>Full results table</b>"))
    display(final_results)


=== Air Quality Prediction System ===

Available example states (first 10):
telangana, the information on this platform is mainly taken from official sources. however, in some cases, a few assumptions have been made and some data derived or assumed and is given in the detailed. while we believe that the data is reliable and adequately comprehensive, niti aayog iced does not take guarantee that such information is in all respects accurate. niti aayog iced does not accept any liability for any consequences resulting from the use of this data.sss, tripura, uttar pradesh, uttarakhand, west bengal



Enter your State:  tripura
Enter your District (press Enter to skip):  



Health Conditions you can choose from:
1. Asthma
2. COPD
3. Chronic bronchitis
4. Cardiovascular disease
5. Smoker
6. Elderly (65+)
7. Child (<12)



Enter condition numbers separated by commas (or press Enter for none):  6



You selected:
State: tripura
District: (none)
Health Conditions: Elderly (65+)


,State,District,Avg AQI,Avg Temp,Avg Rainfall,Avg Wind Speed,Predicted AQI,CHIS,Prescription
677,tripura,dhalai,120.958285,19.871812,123.809804,1.716667,126.613617,0.407722,High risk: Reduce outdoor exertion. Masks reco...


,State,District,Avg AQI,Avg Temp,Min Temp,Max Temp,Avg Rainfall,Max Rainfall,Avg Wind Speed,Min Wind,Max Wind,CHIS,Predicted AQI,Prescription
643,telangana,hyderabad,91.798212,20.115714,17.240000,22.850000,90.891667,268.20000,2.933333,1.900000,5.100000,0.388740,127.467540,High risk: Reduce outdoor exertion. Masks reco...
644,telangana,jagitial,120.958285,19.871812,14.110939,24.331315,123.809804,350.12375,2.283333,1.500000,3.400000,0.450090,113.326951,Moderate risk: Sensitive groups should limit o...
645,telangana,jangaon,120.958285,19.871812,14.110939,24.331315,123.809804,350.12375,2.966667,2.000000,4.900000,0.501180,124.008047,Moderate risk: Sensitive groups should limit o...
646,telangana,jayashankar bhupalapally,120.958285,19.871812,14.110939,24.331315,123.809804,350.12375,1.925000,1.400000,2.600000,0.423298,120.773421,Moderate risk: Sensitive groups should limit o...
647,telangana,jogulamba gadwal,120.958285,19.871812,14.110939,24.331315,123.809804,350.12375,3.050000,2.100000,4.600000,0.507411,127.084258,High risk: Reduce outdoor exertion. Masks reco...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
798,west bengal,purba medinipur,86.586914,19.871812,14.110939,24.331315,123.809804,350.12375,2.508333,2.000000,3.100000,0.347727,131.805421,High risk: Reduce outdoor exertion. Masks reco...
799,west bengal,purba mednapur,120.958285,22.301562,13.890000,27.690000,187.103333,642.10000,2.249381,1.561752,3.117942,0.494092,118.597261,Moderate risk: Sensitive groups should limit o...
800,west bengal,purulia,120.958285,20.021667,11.570000,24.930000,15.625000,33.60000,2.291667,1.500000,2.800000,0.415037,107.539795,Moderate risk: Sensitive groups should limit o...
801,west bengal,south 24 parganas,120.958285,22.352821,14.260000,27.900000,263.358621,849.00000,2.329167,1.800000,3.100000,0.526803,111.385504,Moderate risk: Sensitive groups should limit o...


In [28]:
# Cell 6 — Export full results + personalized summary for the last inputs from Cell 5

import json
from datetime import datetime
from IPython.display import display
import os

# ---- Guards ----
assert 'final_results' in globals(), "final_results not found. Run Cells 1–4 first."
assert 'out' in globals(), "out not found. Run Cells 1–4 first."
assert 'state_in' in globals(), "state_in not found. Run Cell 5 (the input cell) first."
assert 'district_in' in globals(), "district_in not found. Run Cell 5 (the input cell) first."
if 'conditions' not in globals():
    conditions = []

# Optional: save into a subfolder
export_dir = "."
os.makedirs(export_dir, exist_ok=True)

full_csv_path = os.path.join(export_dir, "deep_learning_predictions.csv")
summary_json_path = os.path.join(export_dir, "personalized_summary.json")
summary_csv_path  = os.path.join(export_dir, "personalized_summary.csv")
preview_csv_path  = os.path.join(export_dir, "selection_preview.csv")

# 1) Save the full table
final_results.to_csv(full_csv_path, index=False)

# 2) Recreate the same selection used in Cell 5
key_state = state_in.strip().lower()
key_dist  = district_in.strip().lower()

sel = out[out['State_key'] == key_state].copy()
if key_dist:
    sel = sel[sel['District_key'] == key_dist].copy()

if sel.empty:
    print(f"No rows matched for state='{state_in}', district='{district_in}'.")
    print(f"Saved only the full table to: {full_csv_path}")
else:
    row = sel.iloc[0]
    aqi_pred = float(row['Predicted AQI'])

    # Personalize with your selected conditions
    personalized_aqi, phrs, presc = personalize(aqi_pred, conditions)

    min_temp, max_temp = float(row['Min Temp']), float(row['Max Temp'])
    min_wind, max_wind = float(row['Min Wind']), float(row['Max Wind'])
    rainfall_label = "Rain expected" if float(row['Avg Rainfall']) > 0.1 else "No significant rainfall"

    # Optional: capture model name if available
    mdl = globals().get('model_used', 'unknown')

    summary = {
        "state": row["State"],
        "district": row["District"],
        "conditions": "; ".join(conditions) if conditions else "",
        "predicted_aqi": round(aqi_pred, 1),
        "personalized_aqi": round(personalized_aqi, 1),
        "phrs": int(phrs),
        "prescription": presc,
        "temp_range_c": f"{min_temp:.1f} – {max_temp:.1f}",
        "wind_range_ms": f"{min_wind:.1f} – {max_wind:.1f}",
        "rainfall_status": rainfall_label,
        "model_used": mdl,
        "generated_at": datetime.now().isoformat(timespec="seconds")
    }

    # 3) Save the personalized summary (JSON + CSV)
    with open(summary_json_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    pd.DataFrame([summary]).to_csv(summary_csv_path, index=False)

    # 4) Save a one-row preview of the matched selection
    sel[['State','District','Avg AQI','Avg Temp','Min Temp','Max Temp',
         'Avg Rainfall','Max Rainfall','Avg Wind Speed','Min Wind','Max Wind',
         'Predicted AQI','CHIS','Prescription']].head(1).to_csv(preview_csv_path, index=False)

    print("Saved:")
    print(" -", full_csv_path)
    print(" -", summary_json_path)
    print(" -", summary_csv_path)
    print(" -", preview_csv_path)
    display(pd.DataFrame([summary]))


Saved:
 - .\deep_learning_predictions.csv
 - .\personalized_summary.json
 - .\personalized_summary.csv
 - .\selection_preview.csv


,state,district,conditions,predicted_aqi,personalized_aqi,phrs,prescription,temp_range_c,wind_range_ms,rainfall_status,model_used,generated_at
0,tripura,dhalai,Elderly (65+),126.6,136.6,46,High risk: Reduce outdoor exertion. Masks reco...,14.1 – 24.3,0.9 – 2.4,Rain expected,CNN-LSTM (TensorFlow),2025-11-05T20:18:40


In [26]:
import os
for root, dirs, files in os.walk("exports"):
    for name in files:
        print(os.path.join(root, name))


exports\deep_learning_predictions.csv
exports\personalized_summary.csv
exports\personalized_summary.json
exports\selection_preview.csv


In [27]:
import os
print("Current working directory:", os.getcwd())


Current working directory: C:\Users\ethin
